# Train fold heads

This Kaggle notebook trains five masked attention heads from precomputed DINOv2 feature shards. It does not read DICOM pixels. Attach the code, feature, consensus-label, and fold datasets before running.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/rsna-knee-heads')
PACKAGE = INPUT / 'rsna-knee-code' / 'rsna-knee-abnormality-detection' / 'src'
FEATURE_DIR = INPUT / 'rsna-knee-train-features'
TARGETS = INPUT / 'rsna-knee-consensus' / 'consensus-labels-v1.csv'
FOLDS = INPUT / 'rsna-knee-consensus' / 'folds-v1.csv'
FINGERPRINT = 'REPLACE_WITH_FEATURE_FINGERPRINT'

assert PACKAGE.exists(), PACKAGE
assert TARGETS.is_file(), TARGETS
assert FOLDS.is_file(), FOLDS
assert FINGERPRINT != 'REPLACE_WITH_FEATURE_FINGERPRINT'
FEATURE_SHARDS = sorted(FEATURE_DIR.glob('*.npz'))
assert FEATURE_SHARDS, FEATURE_DIR
WORK.mkdir(parents=True, exist_ok=True)
os.environ['PYTHONPATH'] = str(PACKAGE)
PYTHON = sys.executable

In [ ]:
command = [
    PYTHON, '-m', 'rsna_knee', 'train-fold-heads',
    '--targets', str(TARGETS),
    '--folds', str(FOLDS),
    '--output', str(WORK),
    '--fingerprint', FINGERPRINT,
    '--device', 'cuda',
]
for shard in FEATURE_SHARDS:
    command.extend(['--feature-shard', str(shard)])
subprocess.run(command, check=True)

In [ ]:
required = [WORK / 'metadata.json', WORK / 'oof.csv']
required += [WORK / f'fold-{fold}.pt' for fold in range(5)]
for path in required:
    assert path.is_file(), path
print('Saved fold heads and OOF predictions to', WORK)